<a href="https://colab.research.google.com/github/Mru321/CSI-Cross-Environment-Generalization/blob/main/Data%20Processing%20and%20ML%20model%20training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import h5py
import numpy as np
import pandas as pd

In [ ]:
file_path = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/channel_data/processed_channe14_LOS_cluster1.hdf5"

with h5py.File(file_path, 'r') as f:
    csi = f['mob_csi'][:]

print(csi.shape)

(64, 7939, 52)


In [ ]:
amplitude = np.abs(csi)
phase = np.angle(csi)

print(amplitude.shape)

(64, 7939, 52)


In [ ]:
dataset_path = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/channel_data"

Feature Extraction

In [ ]:
def extract_features(csi):

    amplitude = np.abs(csi)

    feature_list = []

    for frame in range(amplitude.shape[1]):

        frame_data = amplitude[:, frame, :]   # (64 antennas, 52 subcarriers)

        # Basic statistics
        mean_amp = np.mean(frame_data)
        std_amp = np.std(frame_data)
        var_amp = np.var(frame_data)
        max_amp = np.max(frame_data)
        min_amp = np.min(frame_data)

        # Spatial correlation (between antennas)
        antenna_corr_matrix = np.corrcoef(frame_data)
        antenna_corr = np.mean(np.abs(antenna_corr_matrix))

        # Frequency correlation (between subcarriers)
        subcarrier_corr_matrix = np.corrcoef(frame_data.T)
        subcarrier_corr = np.mean(np.abs(subcarrier_corr_matrix))

        feature_vector = [
            mean_amp,
            std_amp,
            var_amp,
            max_amp,
            min_amp,
            antenna_corr,
            subcarrier_corr
        ]

        feature_list.append(feature_vector)

    return np.array(feature_list)

Adding los/nlos labels to dataset according to its environment

In [ ]:
X = []
y = []
env = []

for file in os.listdir(dataset_path):

    if file.endswith(".hdf5"):

        file_path = os.path.join(dataset_path, file)

        print("Processing:", file)

        with h5py.File(file_path, 'r') as f:
            csi = f['mob_csi'][:]

        features = extract_features(csi)

        for feat in features:

            X.append(feat)

            if "NLOS" in file:
                y.append(0)
            elif "LOS" in file:
                y.append(1)

            env.append(file)

Processing: processed_channe14_LOS_cluster1.hdf5
Processing: processed_channe14_LOS_cluster2.hdf5
Processing: processed_channe14_LOS_cluster3.hdf5
Processing: processed_channe14_LOS_cluster4.hdf5
Processing: processed_channe14_NLOS_cluster1.hdf5
Processing: processed_channe14_NLOS_cluster2.hdf5
Processing: processed_channe14_NLOS_cluster3.hdf5
Processing: processed_channe14_NLOS_cluster4.hdf5
Processing: processed_channe14_NLOS_cluster5.hdf5


In [ ]:
columns = [
    "mean_amp",
    "std_amp",
    "var_amp",
    "max_amp",
    "min_amp",
    "antenna_corr",
    "subcarrier_corr"
]

df = pd.DataFrame(X, columns=columns)

df["label"] = y
df["environment"] = env

Saving all the samples in one file

In [ ]:
save_path = "/content/drive/MyDrive/Project/Project1/processed/csi_feature_dataset.csv"

df.to_csv(save_path, index=False)

print("Feature dataset saved to:", save_path)

Feature dataset saved to: /content/drive/MyDrive/Project/Project1/processed/csi_feature_dataset.csv


In [ ]:
df = pd.read_csv(save_path)
df.head()

,mean_amp,std_amp,var_amp,max_amp,min_amp,antenna_corr,subcarrier_corr,label,environment
0,2.488678,0.579163,0.335430,3.975699,0.559248,0.542688,0.815294,1,processed_channe14_LOS_cluster1.hdf5
1,2.488807,0.575728,0.331463,3.883031,0.569000,0.550662,0.816676,1,processed_channe14_LOS_cluster1.hdf5
2,2.489268,0.579336,0.335631,3.982258,0.566464,0.538711,0.818489,1,processed_channe14_LOS_cluster1.hdf5
3,2.489166,0.579777,0.336141,4.024301,0.540201,0.526773,0.818690,1,processed_channe14_LOS_cluster1.hdf5
4,2.489647,0.577595,0.333616,3.971815,0.537490,0.534651,0.819378,1,processed_channe14_LOS_cluster1.hdf5


In [ ]:
df.shape

(53016, 9)

Features of dataset

In [ ]:
df.columns


Index(['mean_amp', 'std_amp', 'var_amp', 'max_amp', 'min_amp', 'antenna_corr',
       'subcarrier_corr', 'label', 'environment'],
      dtype='object')

Count of los/nlos samples

In [ ]:
df["label"].value_counts()

,count
label,
1,31708
0,21308


Training and testing set

In [ ]:
train_env = [
    "LOS_cluster1",
    "LOS_cluster2",
    "LOS_cluster3",
    "NLOS_cluster1",
    "NLOS_cluster2",
    "NLOS_cluster3"
]

test_env = [
    "LOS_cluster4",
    "NLOS_cluster4",
    "NLOS_cluster5"
]

train_mask = df["environment"].str.contains("|".join(train_env))
test_mask = df["environment"].str.contains("|".join(test_env))

train_df = df[train_mask]
test_df = df[test_mask]

Removing label and environment feature from training data

In [ ]:
X_train = train_df.drop(columns=["label", "environment"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label", "environment"])
y_test = test_df["label"]

Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Variance Threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.001)

X_train_sel = selector.fit_transform(X_train_scaled)
X_test_sel = selector.transform(X_test_scaled)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(X_train_sel, y_train)

RandomForestClassifier()

In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test_sel)

accuracy = accuracy_score(y_test, pred)

print("Cross-environment accuracy:", accuracy)

Cross-environment accuracy: 0.998975541042387


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.9979787771601819
Recall: 1.0
F1 Score: 0.9989883662114315


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)

print(cm)

[[7702   16]
 [   0 7900]]


In [ ]:
train_df["environment"].value_counts()

,count
environment,
processed_channe14_LOS_cluster2.hdf5,7958
processed_channe14_LOS_cluster1.hdf5,7939
processed_channe14_LOS_cluster3.hdf5,7911
processed_channe14_NLOS_cluster1.hdf5,5880
processed_channe14_NLOS_cluster3.hdf5,3857
processed_channe14_NLOS_cluster2.hdf5,3853


In [ ]:
test_df["environment"].value_counts()

,count
environment,
processed_channe14_LOS_cluster4.hdf5,7900
processed_channe14_NLOS_cluster5.hdf5,3862
processed_channe14_NLOS_cluster4.hdf5,3856


Count of train label and test label

In [ ]:
print("Train labels")
print(y_train.value_counts())

print("\nTest labels")
print(y_test.value_counts())

Train labels
label
1    23808
0    13590
Name: count, dtype: int64

Test labels
label
1    7900
0    7718
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import accuracy_score

train_pred = model.predict(X_train_scaled)
test_pred  = model.predict(X_test_scaled)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

Train Accuracy: 1.0
Test Accuracy: 0.998975541042387


Feature Importance After Random Forest Training

In [ ]:
import pandas as pd

importance = model.feature_importances_

feature_names = X_train.columns

imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(imp_df)

           Feature  Importance
6  subcarrier_corr    0.413696
4          min_amp    0.215302
5     antenna_corr    0.167164
3          max_amp    0.103178
1          std_amp    0.035053
0         mean_amp    0.033454
2          var_amp    0.032153


SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(random_state=42)

svm_model.fit(X_train_scaled, y_train)

SVC(random_state=42)

In [ ]:
svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred = svm_model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score

print("Train Accuracy:", accuracy_score(y_train, svm_train_pred))
print("Test Accuracy:", accuracy_score(y_test, svm_test_pred))

Train Accuracy: 0.9993047756564523
Test Accuracy: 0.9984633115635805


In [ ]:
from sklearn.metrics import confusion_matrix

cm_svm = confusion_matrix(y_test, svm_test_pred)
print("Confusion Matrix:\n", cm_svm)

Confusion Matrix:
 [[7694   24]
 [   0 7900]]
